In [2]:
import pandas as pd
import pickle
from mlxtend.frequent_patterns import association_rules

## Pattern mining on purchases and different sections

This section of the analysis aims to show the most frequent patterns in purchase products of different sections in the same session. The purpose for this is to show a link to a particular section when the user adds to the basket a product of another section, in order to push the user to buy more products.

In [19]:
with open("frequent_purchase_section_2.pkl", "rb") as f:  # "rb" = read binary
    s_purchases = pickle.load(f)

s_purchases = pd.DataFrame(s_purchases)

print("Number of frequent sections")
print(len(s_purchases))

Number of frequent sections
141


In [45]:
s_purchases_rules = association_rules(s_purchases, metric="confidence", min_threshold=0.3)
s_purchases_rules = s_purchases_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]

print("Association rules with minimum confidence level set to 0.3")
s_purchases_rules

Association rules with minimum confidence level set to 0.3


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"frozenset({ computers, accessories})",frozenset({ appliances}),0.005587,0.436083,0.002456,0.439573,1.008003
1,"frozenset({ accessories, appliances})",frozenset({ computers}),0.006463,0.350288,0.002456,0.379981,1.084767
2,"frozenset({ accessories, electronics})",frozenset({ computers}),0.003549,0.350288,0.001193,0.336283,0.960020
3,"frozenset({ accessories, electronics})",frozenset({ appliances}),0.003549,0.436083,0.001278,0.360177,0.825937
4,frozenset({ electronics}),frozenset({ appliances}),0.123306,0.436083,0.040101,0.325218,0.745770
...,...,...,...,...,...,...,...
78,"frozenset({ construction, sport})",frozenset({ computers}),0.007154,0.350288,0.002453,0.342845,0.978752
79,"frozenset({ computers, construction, sport})",frozenset({ appliances}),0.002453,0.436083,0.001482,0.604353,1.385868
80,"frozenset({ construction, sport, appliances})",frozenset({ computers}),0.003332,0.350288,0.001482,0.444863,1.269994
81,"frozenset({ construction, country_yard})",frozenset({ appliances}),0.009069,0.436083,0.005555,0.612535,1.404629


Using confidence as threshold value to determine the best association rules is not so reliable in this situation. This is because there are some sections (particularly "appliances" and "computers") with a really high support. The computation of confidence takes into account only how frequent the antecedent is, causing an inflation of the measure. This can also be seen when looking at the lift, which for some rules is even below 1. This is why, in this situation, is much more better to compute association rules using a minimum lift threshold.

In [ ]:
s_purchases_rules = association_rules(s_purchases, metric="lift", min_threshold=2)
s_purchases_rules = s_purchases_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]

print("Association rules with minimum lift level set to 2")
s_purchases_rules

Association rules with minimum confidence level set to 0.3


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,"frozenset({ computers, electronics})","frozenset({ sport, appliances})",0.034643,0.043367,0.003505,0.101160,2.332649,1.0,0.002002,1.064297,0.591805,0.047037,0.060413,0.090986
1,"frozenset({ computers, sport})","frozenset({ electronics, appliances})",0.039282,0.040101,0.003505,0.089216,2.224763,1.0,0.001929,1.053926,0.573023,0.046186,0.051166,0.088304
2,"frozenset({ computers, appliances})","frozenset({ electronics, sport})",0.104244,0.013089,0.003505,0.033619,2.568525,1.0,0.002140,1.021244,0.681739,0.030788,0.020802,0.150686
3,"frozenset({ electronics, sport})","frozenset({ computers, appliances})",0.013089,0.104244,0.003505,0.267754,2.568525,1.0,0.002140,1.223299,0.618770,0.030788,0.182539,0.150686
4,"frozenset({ electronics, appliances})","frozenset({ computers, sport})",0.040101,0.039282,0.003505,0.087392,2.224763,1.0,0.001929,1.052718,0.573513,0.046186,0.050078,0.088304
5,"frozenset({ sport, appliances})","frozenset({ computers, electronics})",0.043367,0.034643,0.003505,0.080811,2.332649,1.0,0.002002,1.050226,0.597202,0.047037,0.047824,0.090986
6,"frozenset({ computers, appliances})","frozenset({ sport, furniture})",0.104244,0.005618,0.001394,0.013375,2.380784,1.0,0.000809,1.007862,0.647465,0.012854,0.007801,0.130779
7,"frozenset({ computers, sport})","frozenset({ furniture, appliances})",0.039282,0.017586,0.001394,0.035494,2.018392,1.0,0.000703,1.018568,0.525186,0.025134,0.018230,0.057390
8,"frozenset({ furniture, appliances})","frozenset({ computers, sport})",0.017586,0.039282,0.001394,0.079286,2.018392,1.0,0.000703,1.043449,0.513588,0.025134,0.041640,0.057390
9,"frozenset({ sport, furniture})","frozenset({ computers, appliances})",0.005618,0.104244,0.001394,0.248183,2.380784,1.0,0.000809,1.191455,0.583247,0.012854,0.160690,0.130779


Using the lift to compute association rules causes another problem. The lift is a measure that can be highly inflated when using really rare itemsets. This happens because the denominator is the product of the support measures of the two itemsets. In this situation probably the best strategy is to set minimum threshold values for both the measures.

In [46]:
s_purchases_rules = association_rules(s_purchases, metric="lift", min_threshold=2)
s_purchases_rules = s_purchases_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]
s_purchases_rules = s_purchases_rules[s_purchases_rules["confidence"] > 0.10]
s_purchases_rules = s_purchases_rules.sort_values("lift", ascending=False)
s_purchases_rules = s_purchases_rules.reset_index(drop=True)
s_purchases_rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"frozenset({ computers, medicine})","frozenset({ electronics, appliances})",0.016800,0.040101,0.001878,0.111776,2.787338
1,"frozenset({ computers, construction})","frozenset({ electronics, appliances})",0.015626,0.040101,0.001743,0.111535,2.781345
2,"frozenset({ computers, country_yard})","frozenset({ sport, appliances})",0.017520,0.043367,0.001963,0.112027,2.583230
3,"frozenset({ electronics, sport})","frozenset({ computers, appliances})",0.013089,0.104244,0.003505,0.267754,2.568525
4,"frozenset({ electronics, furniture})","frozenset({ computers, appliances})",0.005637,0.104244,0.001445,0.256267,2.458333
5,"frozenset({ sport, furniture})","frozenset({ computers, appliances})",0.005618,0.104244,0.001394,0.248183,2.380784
6,"frozenset({ electronics, medicine})","frozenset({ computers, appliances})",0.007675,0.104244,0.001878,0.244681,2.347185
7,"frozenset({ computers, electronics})","frozenset({ sport, appliances})",0.034643,0.043367,0.003505,0.101160,2.332649
8,"frozenset({ electronics, construction})","frozenset({ computers, appliances})",0.007176,0.104244,0.001743,0.242888,2.329990
9,"frozenset({ computers, medicine})","frozenset({ sport, appliances})",0.016800,0.043367,0.001686,0.100374,2.314514


### Conclusions

The rules above should suggest possible purchase patterns followed by the users. When an user buys one or more products of the antecedent section / sections, there is a reasonable possibility that he will buy one or more product also of the consequent section / sections.
The advised strategy is set as follows:
- Let's take as an example the first rule, so {computers, medicine} -> {electronics, appliances};
- Let's assume an user has already added to its basket one product from "computers" section and one from "medicine" section;
- After the user adds to the basket this last product, the page should show a fast link to the "electronics" section and one to the "appliances" section.

## Pattern mining on purchases and products

The purpose of this section is to find strong associations between purchases of products, in order to provide an efficient advise policy and encourage the user to buy more products.

In [39]:
with open("frequent_purchases.pkl", "rb") as f:  # "rb" = read binary
    purchases = pickle.load(f)

purchases = pd.DataFrame(purchases)

print("Number of frequent itemsets")
print(len(purchases))

Number of frequent itemsets
524


In [62]:
purchases_rules = association_rules(purchases, metric="confidence", min_threshold=0.4)
purchases_rules = purchases_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]
print("Association rules with minimum confidence level set to 0.4")
purchases_rules

Association rules with minimum confidence level set to 0.4


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,frozenset({214820261}),frozenset({214820233}),0.005524,0.003407,0.002283,0.413303,121.303101
1,frozenset({214820233}),frozenset({214820261}),0.003407,0.005524,0.002283,0.670046,121.303101
2,frozenset({214716710}),frozenset({214716707}),0.001928,0.003583,0.001212,0.628664,175.455247
3,frozenset({214834880}),frozenset({214834877}),0.010328,0.010419,0.004491,0.434783,41.728123
4,frozenset({214834877}),frozenset({214834880}),0.010419,0.010328,0.004491,0.430983,41.728123
5,frozenset({214834880}),frozenset({214826610}),0.010328,0.009468,0.004924,0.476741,50.353300
6,frozenset({214826610}),frozenset({214834880}),0.009468,0.010328,0.004924,0.520066,50.353300
7,frozenset({214826610}),frozenset({214834877}),0.009468,0.010419,0.004098,0.432836,41.541281
8,"frozenset({214834880, 214834877})",frozenset({214826610}),0.004491,0.009468,0.002371,0.527972,55.764353
9,"frozenset({214834880, 214826610})",frozenset({214834877}),0.004924,0.010419,0.002371,0.481505,46.212300


In [59]:
purchases_rules_no = association_rules(purchases, metric="lift", min_threshold=0.0000000001)
print("Number of association rules with no threshold value")
print(len(purchases_rules_no))
purchases_rules_l = association_rules(purchases, metric="lift", min_threshold=2)
print("Number of patterns with threshold value of the lift set equal to 1.5")
print(len(purchases_rules_l))

Number of association rules with no threshold value
216
Number of patterns with threshold value of the lift set equal to 1.5
216


# Conclusions

In this situation the confidence is a more accurate measure because the lift is highly inflated because the itemsets are not really frequent. This is easily showed by the fact that the number of association rules does not change with or without a normal threshold value for the lift (2 should already be a "good" association).

The advised strategy is to show a link / advertisement for the consequent product / products when the antecedent product / products are added to the basket.

In [63]:
purchases_rules = purchases_rules[["antecedents", "consequents", "antecedent support", "consequent support", "support", "confidence", "lift"]]
mono_associations = purchases_rules[(purchases_rules["antecedents"].apply(len) == 1) & (purchases_rules["consequents"].apply(len) == 1)]
multi_associations = purchases_rules[(purchases_rules["antecedents"].apply(len) > 1) | (purchases_rules["consequents"].apply(len) > 1)]

print(mono_associations.head())
print(multi_associations)

              antecedents             consequents  antecedent support  \
0  frozenset({214820261})  frozenset({214820233})            0.005524   
1  frozenset({214820233})  frozenset({214820261})            0.003407   
2  frozenset({214716710})  frozenset({214716707})            0.001928   
3  frozenset({214834880})  frozenset({214834877})            0.010328   
4  frozenset({214834877})  frozenset({214834880})            0.010419   

   consequent support   support  confidence        lift  
0            0.003407  0.002283    0.413303  121.303101  
1            0.005524  0.002283    0.670046  121.303101  
2            0.003583  0.001212    0.628664  175.455247  
3            0.010419  0.004491    0.434783   41.728123  
4            0.010328  0.004491    0.430983   41.728123  
                          antecedents             consequents  \
8   frozenset({214834880, 214834877})  frozenset({214826610})   
9   frozenset({214834880, 214826610})  frozenset({214834877})   
10  frozenset({214

In [ ]:
purchases_rules["pair"] = purchases_rules.apply(
    lambda r: tuple(sorted([
        next(iter(r["antecedents"])),
        next(iter(r["consequents"]))
    ])),
    axis=1
)
pair_counts = purchases_rules["pair"].value_counts()
asymmetric_pairs = pair_counts[pair_counts == 1]
print("Mono-directional relationships")
asymmetric_pairs

pair
(214716707, 214716710)    1
(214826610, 214834877)    1
(214833755, 214834877)    1
(214826610, 214833755)    1
(214826610, 214829387)    1
(214829387, 214834880)    1
(214829387, 214834877)    1
(214829861, 214829865)    1
(214829846, 214829865)    1
(214829878, 214829882)    1
(214829885, 214831950)    1
(214835017, 214835019)    1
(214835109, 214836932)    1
(214826990, 214836932)    1
(214835109, 214838227)    1
(214826990, 214838227)    1
(214839971, 214839973)    1
(214837286, 214838092)    1
(214844355, 214844370)    1
(214848337, 214848380)    1
Name: count, dtype: int64

For an association rule {A} -> {B} that respects the confidence threshold value let's define it as:
- Bi-directional rule: if the relation {B} -> {A} also respects the confidence threshold value;
- Mono-directional rule: if the relation {B} -> {A} does not respect the confidence threshold value.

The last output shows all the mono-directional rules regarding purchases with only one product as antecedent (the first one) and only one product as consequent (the second one). The business strategy of above suggests to show the link / advertisement only when the first product is added to the basket, but another strategy could be implemented for the opposite relation.

Assuming the rule to be {A} -> {B}, let's assume that in general it is really likely to buy the two products together, but the customer is likely to remember to buy product B after buying product A, but not to buy product A after buying product B. After implementing a correct A/B test to check for this hypothesis, an efficient strategy would be to show advertiments for both the directions, in order to increase purchases of the products together, independently of which one is bought first.

## Pattern mining on clicks and products

In [ ]:
with open("frequent_clicks.pkl", "rb") as f:  # "rb" = read binary
    clicks = pickle.load(f)

clicks = pd.DataFrame(clicks)

clicks_rules = association_rules(clicks, metric="confidence", min_threshold=0.2)
print(len(clicks_rules))

In [ ]:
clicks_rules = clicks_rules[["antecedents", "consequents", "antecedent support", "consequent support", "support", "confidence", "lift"]]
mono_associations = clicks_rules[(clicks_rules["antecedents"].apply(len) == 1) & (clicks_rules["consequents"].apply(len) == 1)]
multi_associations = clicks_rules[(clicks_rules["antecedents"].apply(len) > 1) | (clicks_rules["consequents"].apply(len) > 1)]

print(mono_associations.head())
print(multi_associations)

In [ ]:
clicks_rules["pair"] = clicks_rules.apply(
    lambda r: tuple(sorted([
        next(iter(r["antecedents"])),
        next(iter(r["consequents"]))
    ])),
    axis=1
)
pair_counts = clicks_rules["pair"].value_counts()
asymmetric_pairs = pair_counts[pair_counts == 1]
asymmetric_pairs